# CodeLens AI — Phase 2: Dataset Preparation

**Course:** CSC-233 AI Lab, Spring 2026 — BNU  
**What this does:** Builds the instruction-following dataset for fine-tuning Gemma 3 4B.

### Steps
1. Install deps + mount Google Drive
2. Clone the repo (to get the generation scripts)
3. Generate 500+ manual translation pairs (Python→Rust, JS→Rust, Python→C++, etc.)
4. Download CodeSearchNet (2M+ pairs) and format as instruction triples
5. Download code_x_glue translation pairs (Python↔Java)
6. Mix 70/30 and split 80/10/10
7. Save train/val/test to Google Drive

**Just run all cells top-to-bottom. Nothing to write manually.**

---
## 0. Setup

In [ ]:
!pip install -q datasets huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random

OUTPUT_DIR = '/content/drive/MyDrive/codelens-ai/finetune/data'
os.makedirs(f'{OUTPUT_DIR}/raw', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/processed', exist_ok=True)

SEED = 42
random.seed(SEED)
print('Ready. Output dir:', OUTPUT_DIR)

In [ ]:
# Clone the repo so we can use the generation scripts
import os
if not os.path.exists('/content/codelens-ai'):
    !git clone https://github.com/MehkaanKhan/codelens-ai.git /content/codelens-ai
else:
    !cd /content/codelens-ai && git pull
print('Repo ready at /content/codelens-ai')

---
## Step 1 — Generate Manual Translation Pairs

Runs the generation script to produce 500+ hand-crafted cross-language pairs:
- Python → Rust (generators/iterators)
- Python → C++ (classes/structs)
- JavaScript → Rust (async/await → tokio futures)
- Python → Go (list comprehensions → slices)
- JavaScript → Java (Promises → CompletableFuture)

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'finetune/scripts/generate_translation_pairs.py'],
    cwd='/content/codelens-ai',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Copy to Drive so it's backed up
import shutil
src = '/content/codelens-ai/finetune/data/raw/manual_translations.jsonl'
dst = f'{OUTPUT_DIR}/raw/manual_translations.jsonl'
shutil.copy(src, dst)

with open(dst) as f:
    count = sum(1 for _ in f)
print(f'Manual pairs copied to Drive: {count} pairs')

---
## Step 2 — Download CodeSearchNet

Dataset: [`code-search-net/code_search_net`](https://huggingface.co/datasets/code-search-net/code_search_net)  
~2.07M function + docstring pairs across Python, JavaScript, Java, Go, PHP, Ruby.  
No token needed — public dataset.

In [ ]:
from datasets import load_dataset

LANGUAGES  = ['python', 'javascript', 'java', 'go', 'php', 'ruby']
DATASET_ID = 'code-search-net/code_search_net'
RAW_OUTPUT = f'{OUTPUT_DIR}/raw/codesearchnet_filtered.jsonl'

total_written = 0
with open(RAW_OUTPUT, 'w', encoding='utf-8') as out_f:
    for lang in LANGUAGES:
        print(f'[{lang}] Loading...')
        ds = load_dataset(DATASET_ID, lang, split='train+validation+test')
        lang_count = 0
        for row in ds:
            code      = (row.get('func_code_string') or '').strip()
            docstring = (row.get('func_documentation_string') or '').strip()
            if not code or not docstring:
                continue
            out_f.write(json.dumps({
                'language': lang,
                'func_name': row.get('func_name', ''),
                'code': code,
                'docstring': docstring,
            }, ensure_ascii=False) + '\n')
            lang_count += 1
        total_written += lang_count
        print(f'  {lang_count:,} samples')

print(f'\nTotal: {total_written:,} -> {RAW_OUTPUT}')

---
## Step 3 — Format CodeSearchNet as Instruction Triples

Converts each function+docstring into:
```json
{"system_prompt": "You are a code summarizer...",
 "user_input": "Summarize this python function: ...```python\n<code>\n```",
 "assistant_output": "<plain-English summary>"}
```

In [ ]:
SUMMARIZATION_SYSTEM = (
    'You are a code summarizer. Given a function or code snippet, '
    'produce a clear, concise plain-English summary of what it does. '
    'Do not describe the syntax — explain the purpose and behavior.'
)
SUMM_OUTPUT = f'{OUTPUT_DIR}/processed/summarization_triples.jsonl'

written = skipped = 0
with open(RAW_OUTPUT, 'r', encoding='utf-8') as in_f, \
     open(SUMM_OUTPUT, 'w', encoding='utf-8') as out_f:
    for line in in_f:
        row  = json.loads(line)
        code = row['code']; doc = row['docstring']; lang = row['language']
        if len(code) < 30 or len(doc) < 10 or len(code) > 3000:
            skipped += 1; continue
        out_f.write(json.dumps({
            'system_prompt': SUMMARIZATION_SYSTEM,
            'user_input': f'Summarize this {lang} function:\n\n```{lang}\n{code}\n```',
            'assistant_output': doc,
        }, ensure_ascii=False) + '\n')
        written += 1

print(f'Written: {written:,} | Skipped: {skipped:,}')
print(f'Saved -> {SUMM_OUTPUT}')

---
## Step 4 — Download HuggingFace Translation Pairs

Loads `code_x_glue_cc_code_to_code_trans` (Python ↔ Java parallel corpus) and combines with the generated manual pairs.

In [ ]:
TRANSLATION_SYSTEM = (
    'You are a code translator. Convert the given code to the target language '
    'while preserving logic and semantics. Use idiomatic style for the target language — '
    'do not just transliterate syntax. Add a short comment where the translation is non-obvious.'
)
TRANS_OUTPUT = f'{OUTPUT_DIR}/processed/translation_triples.jsonl'

def make_triple(src_lang, tgt_lang, src_code, tgt_code):
    return {
        'system_prompt': TRANSLATION_SYSTEM,
        'user_input': f'Convert this {src_lang} code to {tgt_lang}:\n\n```{src_lang.lower()}\n{src_code.strip()}\n```',
        'assistant_output': f'```{tgt_lang.lower()}\n{tgt_code.strip()}\n```',
    }

triples = []

# Part A: HuggingFace Python<->Java corpus
print('Loading code_x_glue...')
try:
    trans_ds = load_dataset('code_x_glue_cc_code_to_code_trans',
                            split='train+validation+test')
    # Inspect actual column names — trust_remote_code is deprecated, dataset now loads from Parquet
    print(f'  Columns: {trans_ds.column_names}')
    sample_row = next(iter(trans_ds))
    print(f'  Sample keys: {list(sample_row.keys())}')

    hf_before = len(triples)
    for row in trans_ds:
        # Field names vary by dataset version — try both casings
        java = (row.get('java') or row.get('Java') or '').strip()
        py   = (row.get('python') or row.get('Python') or '').strip()
        if java and py:
            triples.append(make_triple('Java',   'Python', java, py))
            triples.append(make_triple('Python', 'Java',   py,  java))
    print(f'  HuggingFace pairs added: {len(triples) - hf_before:,} (both directions)')
except Exception as e:
    print(f'  WARNING: {e} — continuing with manual pairs only')

# Part B: Generated manual pairs
manual_path = f'{OUTPUT_DIR}/raw/manual_translations.jsonl'
manual_count = 0
with open(manual_path, 'r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if all(k in row for k in ('system_prompt', 'user_input', 'assistant_output')):
            triples.append(row); manual_count += 1
print(f'  Manual/generated pairs: {manual_count:,}')
print(f'\nTotal translation triples: {len(triples):,}')

with open(TRANS_OUTPUT, 'w', encoding='utf-8') as f:
    for t in triples:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')
print(f'Saved -> {TRANS_OUTPUT}')

---
## Step 5 — Mix 70/30 and Split 80/10/10

Mixes summarization and translation samples at the right ratio, then splits into train/val/test.

In [ ]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def write_jsonl(path, records):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + '\n')

summ  = load_jsonl(SUMM_OUTPUT)
trans = load_jsonl(TRANS_OUTPUT)
print(f'Summarization : {len(summ):,}')
print(f'Translation   : {len(trans):,}')

# Balance to 70/30 without over-sampling either side
needed_summ = int(len(trans) * (0.70 / 0.30))
if needed_summ <= len(summ):
    sampled_summ  = random.sample(summ, needed_summ)
    sampled_trans = trans
else:
    sampled_summ  = summ
    sampled_trans = random.sample(trans, min(int(len(summ) * (0.30 / 0.70)), len(trans)))

mixed = sampled_summ + sampled_trans
random.shuffle(mixed)

n = len(mixed)
train_end = int(n * 0.80)
val_end   = train_end + int(n * 0.10)
train = mixed[:train_end]
val   = mixed[train_end:val_end]
test  = mixed[val_end:]

summ_in_train  = sum(1 for r in train if 'Summarize' in r.get('user_input', ''))
trans_in_train = len(train) - summ_in_train

print(f'\nMixed : {len(mixed):,}')
print(f'  Train : {len(train):,}  ({len(train)/len(mixed)*100:.1f}%)')
print(f'  Val   : {len(val):,}  ({len(val)/len(mixed)*100:.1f}%)')
print(f'  Test  : {len(test):,}  ({len(test)/len(mixed)*100:.1f}%)')
print(f'\nTrain task split:')
print(f'  Summarization : {summ_in_train:,}  ({summ_in_train/len(train)*100:.1f}%)')
print(f'  Translation   : {trans_in_train:,}  ({trans_in_train/len(train)*100:.1f}%)')

write_jsonl(f'{OUTPUT_DIR}/processed/train.jsonl', train)
write_jsonl(f'{OUTPUT_DIR}/processed/val.jsonl',   val)
write_jsonl(f'{OUTPUT_DIR}/processed/test.jsonl',  test)

print(f'\nSaved to Drive: {OUTPUT_DIR}/processed/')

---
## Step 6 — Verify

In [ ]:
for split_name in ['train', 'val', 'test']:
    path  = f'{OUTPUT_DIR}/processed/{split_name}.jsonl'
    lines = load_jsonl(path)
    ok    = all(k in lines[0] for k in ('system_prompt', 'user_input', 'assistant_output'))
    print(f'{split_name:5s}: {len(lines):>8,} samples | format OK: {ok}')

print('\nSample:')
s = load_jsonl(f'{OUTPUT_DIR}/processed/train.jsonl')[0]
print(f'  system_prompt   : {s["system_prompt"][:60]}...')
print(f'  user_input      : {s["user_input"][:80]}...')
print(f'  assistant_output: {s["assistant_output"][:80]}...')

---
## Done — Handoff to Mehkaan

Share this folder with Mehkaan for Phase 3:
`My Drive/codelens-ai/finetune/data/processed/`

Files to share:
- `train.jsonl`
- `val.jsonl`  
- `test.jsonl`